# 🤖 AI Engineering Fundamentals — Lezione 6
## Notebook Gruppo A

**ITS Novitas 4.0 | Martedì 09/06/2026 🏁**

---

### 📋 Istruzioni
1. **File → Salva una copia in Drive** prima di iniziare
2. Lavorate in gruppo — discutete prima di scrivere
3. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo

In [ ]:
GRUPPO = "A"
MEMBRI = ["Alfonso Mammato"]  # ← inserite i vostri nomi
print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

In [2]:
# Setup — eseguite questa cella per prima
# La API key viene letta dal file .env nella root del progetto (non più dai Secrets di Colab)
# streamlit è già nel requirements.txt
import anthropic, os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))   # carica ANTHROPIC_API_KEY dal .env

client = anthropic.Anthropic()          # legge automaticamente ANTHROPIC_API_KEY dall'ambiente
print("✅ Setup completato!")

✅ Setup completato!


---
## 🎯 Tema del Gruppo A: Streamlit — Costruire la UI

Esplorate i componenti Streamlit per costruire
una chat UI professionale. Il vostro obiettivo è
capire il meccanismo di rerun e il session state.

---
### Esercizio 1 — Scrivere l'app base *(guidato)*

Scrivete il file `app_base.py` — la chat UI minima funzionante.
Poi eseguitela con `streamlit run` tramite ngrok per vederla nel browser.

In [ ]:
# Esercizio 1 — app Streamlit base
# Scrivete il file su disco, poi lo eseguiamo

app_base = '''
import streamlit as st
import anthropic
import os
from dotenv import load_dotenv, find_dotenv

# Carica la API key dal .env nella root del progetto
load_dotenv(find_dotenv(usecwd=True))

# Configurazione pagina
st.set_page_config(
    page_title="Chatbot WiData",
    page_icon="🤖",
    layout="centered"
)

client = anthropic.Anthropic()

SYSTEM = "Sei l'assistente di WiData Srl, startup IoT di Sassari."

# 1) Inizializziamo il session state per la history
if "messages" not in st.session_state:
    st.session_state.messages = []

st.title("🤖 Chatbot WiData")
st.caption("Assistente virtuale per prodotti IoT e smart cities")

# 2) Mostriamo i messaggi della history
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# 3) Gestiamo l'input utente e generiamo la risposta (in streaming)
if prompt := st.chat_input("Scrivi un messaggio..."):
    # mostra e salva il messaggio utente
    with st.chat_message("user"):
        st.markdown(prompt)
    st.session_state.messages.append({"role": "user", "content": prompt})

    # genera la risposta dell'assistente
    with st.chat_message("assistant"):
        risposta = ""
        placeholder = st.empty()
        with client.messages.stream(
            model="claude-haiku-4-5-20251001",
            max_tokens=500,
            system=SYSTEM,
            messages=st.session_state.messages,
        ) as stream:
            for text in stream.text_stream:
                risposta += text
                placeholder.markdown(risposta + "▌")
        placeholder.markdown(risposta)

    st.session_state.messages.append({"role": "assistant", "content": risposta})
'''

with open("app_base.py", "w", encoding="utf-8") as f:
    f.write(app_base)

print("✅ app_base.py creato")
print()
print("Eseguire: streamlit run app_base.py")

In [ ]:
"""# Esegui l'app con ngrok per vederla nel browser
#!pip install pyngrok -q
from pyngrok import ngrok
import subprocess, time

# Avvia streamlit in background
proc = subprocess.Popen(
    ["streamlit", "run", "app_base.py",
     "--server.port", "8501",
     "--server.headless", "true"],
    env={**os.environ}
)
time.sleep(3)

# Crea tunnel pubblico
tunnel = ngrok.connect(8501)
print(f"🌐 App online: {tunnel.public_url}")
print("Aprite il link nel browser. Ctrl+C per fermare.")
"""

---
### Esercizio 2 — Session state e rerun *(guidato)*

Dimostrate sperimentalmente il meccanismo di rerun.
Aggiungete un contatore nella sidebar che si incrementa
ad ogni interazione — mostra quante volte Streamlit
ha rieseguito lo script.

In [ ]:
# Esercizio 2 — dimostrare il meccanismo di rerun

app_rerun = '''
import streamlit as st
import anthropic, os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

client = anthropic.Anthropic()
st.set_page_config(page_title="Demo Rerun", page_icon="🔄")

# Contatore di rerun — dimostra che Streamlit riesegue tutto
if "rerun_count" not in st.session_state:
    st.session_state.rerun_count = 0
if "messages" not in st.session_state:
    st.session_state.messages = []

# Incrementiamo il contatore ad OGNI esecuzione dello script (cioè ad ogni rerun)
st.session_state.rerun_count += 1

# Sidebar con informazioni di debug
with st.sidebar:
    st.title("🔍 Debug")
    st.metric("Numero di rerun", st.session_state.rerun_count)
    st.metric("Messaggi in storia", len(st.session_state.messages))
    st.divider()
    if st.button("🗑️ Reset storia"):
        st.session_state.messages = []
        st.rerun()

st.title("Demo Rerun — Streamlit")
st.info("Ogni volta che invii un messaggio, Streamlit riesegue"
        " TUTTO lo script. Il session state mantiene i dati tra i rerun.")

# Chat UI
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if prompt := st.chat_input("Scrivi per vedere il rerun..."):
    with st.chat_message("user"):
        st.markdown(prompt)
    st.session_state.messages.append({"role": "user", "content": prompt})

    with st.chat_message("assistant"):
        risposta = ""
        placeholder = st.empty()
        with client.messages.stream(
            model="claude-haiku-4-5-20251001",
            max_tokens=300,
            messages=st.session_state.messages
        ) as stream:
            for text in stream.text_stream:
                risposta += text
                placeholder.markdown(risposta + "▌")
        placeholder.markdown(risposta)

    st.session_state.messages.append({"role": "assistant", "content": risposta})
'''

with open("app_rerun.py", "w", encoding="utf-8") as f:
    f.write(app_rerun)

print("✅ app_rerun.py creato")
print("Avviatela con: streamlit run app_rerun.py")

✅ app_rerun.py creato
Avviatela con ngrok (cella sotto) o in locale: streamlit run app_rerun.py


---
### Esercizio 3 — Componenti avanzati *(libero)*

Aggiungete all'app base questi componenti:
- `st.file_uploader()` per caricare un PDF
- `st.slider()` per controllare la temperature
- `st.metric()` per mostrare token usati e costo
- `st.feedback()` per thumbs up/down su ogni risposta

Ognuno in una cella separata — testate uno per volta.

In [ ]:
# Esercizio 3 — app con componenti avanzati

app_avanzata = '''
import streamlit as st
import anthropic, os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

client = anthropic.Anthropic()
st.set_page_config(page_title="Chatbot WiData Pro", page_icon="🤖", layout="wide")

# Session state
if "messages" not in st.session_state:
    st.session_state.messages = []
if "token_totali" not in st.session_state:
    st.session_state.token_totali = 0

# ── Sidebar ──────────────────────────────────────────────────────────
with st.sidebar:
    st.title("⚙️ Impostazioni")

    # Slider per la temperature
    temperature = st.slider("Temperature", 0.0, 1.0, 0.7)

    # Slider per max_tokens
    max_tokens = st.slider("Max tokens", 100, 1000, 500)

    st.divider()

    # File uploader per PDF (in questo esercizio lo mostriamo soltanto)
    uploaded = st.file_uploader("📄 Carica PDF", type="pdf")
    if uploaded:
        st.success(f"Caricato: {uploaded.name}")

    st.divider()

    # Metriche: token totali e costo stimato (Haiku: $1 / milione di token)
    costo = st.session_state.token_totali / 1_000_000 * 1.0
    st.metric("Token usati", st.session_state.token_totali)
    st.metric("Costo stimato", f"${costo:.5f}")

    st.divider()
    if st.button("🗑️ Nuova chat"):
        st.session_state.messages = []
        st.session_state.token_totali = 0
        st.rerun()

# ── Main ──────────────────────────────────────────────────────────────
st.title("🤖 Chatbot WiData Pro")

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if prompt := st.chat_input("Scrivi un messaggio..."):
    with st.chat_message("user"):
        st.markdown(prompt)
    st.session_state.messages.append({"role": "user", "content": prompt})

    with st.chat_message("assistant"):
        risposta = ""
        placeholder = st.empty()
        with client.messages.stream(
            model="claude-haiku-4-5-20251001",
            max_tokens=max_tokens,
            temperature=temperature,
            messages=st.session_state.messages
        ) as stream:
            for text in stream.text_stream:
                risposta += text
                placeholder.markdown(risposta + "▌")
        placeholder.markdown(risposta)

        # Feedback thumbs up/down sotto la risposta
        st.feedback("thumbs")

    st.session_state.messages.append({"role": "assistant", "content": risposta})
    # Stima dei token usati dalla risposta (~4 caratteri per token)
    st.session_state.token_totali += len(risposta) // 4
'''

with open("app_avanzata.py", "w", encoding="utf-8") as f:
    f.write(app_avanzata)

print("✅ app_avanzata.py creato")
print("Avviatela con ngrok o in locale: streamlit run app_avanzata.py")

✅ app_avanzata.py creato
Avviatela con ngrok o in locale: streamlit run app_avanzata.py


---
### Esercizio 4 — App completa con RAG *(libero)*

Integrate ChromaDB nell'app Streamlit usando `@st.cache_resource`
per non reinizializzare il database ad ogni rerun.
L'utente deve poter caricare un PDF e chattare su di esso.

In [5]:
# Esercizio 4 — app con RAG integrato

# Esercizio 4 — app con RAG integrato

app_rag = '''
import io
import streamlit as st
import anthropic
import chromadb

from pypdf import PdfReader
from dotenv import load_dotenv, find_dotenv
from chromadb.utils import embedding_functions

load_dotenv(find_dotenv(usecwd=True))

client = anthropic.Anthropic()

st.set_page_config(
    page_title="Chatbot RAG WiData",
    page_icon="🤖",
    layout="wide"
)

SYSTEM = """
Sei l'assistente di WiData Srl.

Rispondi esclusivamente utilizzando le informazioni presenti
nel contesto fornito.

Se la risposta non è contenuta nei documenti, dichiara
esplicitamente che non possiedi tale informazione.
"""

# --------------------------------------------------
# ChromaDB
# --------------------------------------------------

@st.cache_resource
def get_chroma_client():
    return chromadb.Client()

@st.cache_resource
def get_embedding_function():
    return embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )

# --------------------------------------------------
# Chunking
# --------------------------------------------------

def chunka(testo, size=400, overlap=50):
    chunks = []

    start = 0

    while start < len(testo):
        chunk = testo[start:start + size]

        if chunk.strip():
            chunks.append(chunk)

        start += size - overlap

    return chunks

# --------------------------------------------------
# PDF → Chroma
# --------------------------------------------------

def indicizza_pdf(file_bytes):

    reader = PdfReader(io.BytesIO(file_bytes))

    testo = " ".join(
        page.extract_text() or ""
        for page in reader.pages
    )

    chunks = chunka(testo)

    chroma = get_chroma_client()

    try:
        chroma.delete_collection("rag_collection")
    except Exception:
        pass

    collection = chroma.create_collection(
        name="rag_collection",
        embedding_function=get_embedding_function()
    )

    collection.add(
        documents=chunks,
        ids=[str(i) for i in range(len(chunks))]
    )

    return collection, len(chunks)

# --------------------------------------------------
# Session State
# --------------------------------------------------

if "messages" not in st.session_state:
    st.session_state.messages = []

if "collection" not in st.session_state:
    st.session_state.collection = None

if "token_totali" not in st.session_state:
    st.session_state.token_totali = 0

# --------------------------------------------------
# Sidebar
# --------------------------------------------------

with st.sidebar:

    st.title("⚙️ Impostazioni")

    temperature = st.slider(
        "Temperature",
        min_value=0.0,
        max_value=1.0,
        value=0.7
    )

    max_tokens = st.slider(
        "Max tokens",
        min_value=100,
        max_value=1000,
        value=500
    )

    st.divider()

    uploaded = st.file_uploader(
        "📄 Carica PDF",
        type="pdf"
    )

    if uploaded:

        with st.spinner("Indicizzazione PDF..."):

            collection, n_chunks = indicizza_pdf(
                uploaded.read()
            )

            st.session_state.collection = collection

        st.success(
            f"PDF caricato: {uploaded.name}"
        )

        st.info(
            f"Chunk indicizzati: {n_chunks}"
        )

    st.divider()

    costo = (
        st.session_state.token_totali
        / 1_000_000
        * 1.0
    )

    st.metric(
        "Token usati",
        st.session_state.token_totali
    )

    st.metric(
        "Costo stimato",
        f"${costo:.5f}"
    )

    st.divider()

    if st.button("🗑️ Nuova chat"):

        st.session_state.messages = []
        st.session_state.collection = None
        st.session_state.token_totali = 0

        st.rerun()

# --------------------------------------------------
# Main
# --------------------------------------------------

st.title("🤖 Chatbot RAG WiData")

if st.session_state.collection is None:
    st.info(
        "Carica un PDF dalla sidebar per iniziare."
    )

# Storico chat

for msg in st.session_state.messages:

    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# --------------------------------------------------
# Input utente
# --------------------------------------------------

if prompt := st.chat_input("Scrivi un messaggio..."):

    with st.chat_message("user"):
        st.markdown(prompt)

    st.session_state.messages.append(
        {
            "role": "user",
            "content": prompt
        }
    )

    # ----------------------------------------------
    # Retrieval
    # ----------------------------------------------

    chunks_trovati = []

    if st.session_state.collection:

        n_results = min(
            3,
            st.session_state.collection.count()
        )

        if n_results > 0:

            risultati = (
                st.session_state.collection.query(
                    query_texts=[prompt],
                    n_results=n_results
                )
            )

            chunks_trovati = risultati["documents"][0]

    # ----------------------------------------------
    # Context building
    # ----------------------------------------------

    if chunks_trovati:

        contesto = "\\n\\n---\\n\\n".join(
            chunks_trovati
        )

        messaggio_rag = f"""
Contesto:

{contesto}

---

Domanda:

{prompt}
"""

    else:

        messaggio_rag = prompt

    history_rag = (
        st.session_state.messages[:-1]
        +
        [{
            "role": "user",
            "content": messaggio_rag
        }]
    )

    # ----------------------------------------------
    # Claude
    # ----------------------------------------------

    with st.chat_message("assistant"):

        risposta = ""

        placeholder = st.empty()

        with client.messages.stream(
            model="claude-haiku-4-5-20251001",
            system=SYSTEM,
            messages=history_rag,
            max_tokens=max_tokens,
            temperature=temperature
        ) as stream:

            for text in stream.text_stream:

                risposta += text

                placeholder.markdown(
                    risposta + "▌"
                )

        placeholder.markdown(risposta)

        if chunks_trovati:

            with st.expander(
                f"📄 Chunk RAG utilizzati ({len(chunks_trovati)})"
            ):

                for i, chunk in enumerate(
                    chunks_trovati,
                    start=1
                ):

                    st.markdown(
                        f"**Chunk {i}**"
                    )

                    st.caption(
                        chunk[:500]
                    )

    st.session_state.messages.append(
        {
            "role": "assistant",
            "content": risposta
        }
    )

    # stima semplice token

    st.session_state.token_totali += (
        len(risposta) // 4
    )
'''
with open("app_rag.py", "w", encoding="utf-8") as f:
    f.write(app_rag)

print("✅ app_rag.py creato!")
print()
print("Questa è la vostra app finale in locale: streamlit run app_rag.py")

✅ app_rag.py creato!

Questa è la vostra app finale in locale: streamlit run app_rag.py


---
## 📊 Preparate la presentazione (5 slide)

1. **Cos'è Streamlit** — il meccanismo di rerun con il contatore
2. **Session state** — perché è fondamentale, cosa succede senza
3. **I componenti chiave** — slider, file_uploader, metric, feedback
4. **@st.cache_resource** — perché serve con ChromaDB
5. **Demo live** — mostrate l'app RAG con un PDF caricato

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*